# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets by @id
print("Available record sets:")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}, description: {record_set.description}")

# Show fields and their @id for each record set
for record_set in metadata.record_sets:
    print(f"\nFields in record set '{record_set.name}' (@id: {record_set.id}):")
    for field in record_set.fields:
        print(f"  * @id: {field.id}, name: {field.name}, data_type: {field.data_type}, description: {field.description}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis using the record set and field `@id`s from the overview.

In [ ]:
# Select the main record set for protein analysis (usually contains core data)
# We'll look for the main tabular record set from the overview above.

main_record_set_id = None
for record_set in metadata.record_sets:
    # Example heuristic: look for record sets with 'protein' or 'table' in the name/description
    name_desc = (record_set.name or '').lower() + (record_set.description or '').lower()
    if 'protein' in name_desc or 'table' in name_desc or main_record_set_id is None:
        main_record_set_id = record_set.id
        # Prefer 'protein' if found
        if 'protein' in name_desc:
            break

# List all record set @id
record_sets_ids = [rs.id for rs in metadata.record_sets]
print("Record sets to extract:", record_sets_ids)

# Load all record sets into dataframes
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Print main table columns
print(f"Columns in DataFrame for record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis based on the record set fields
main_field_ids = {f.id: f for f in [
    field for rs in metadata.record_sets if rs.id==main_record_set_id for field in rs.fields
]} if main_record_set_id else {}

# Try to guess a good numeric field, e.g., fold change, abundance, coverage, peptide_count
selected_numeric_field = None
for fid, f in main_field_ids.items():
    f_name = (f.name or '').lower()
    if any(x in f_name for x in ["abundance", "coverage", "count", "fold", "mw", "weight"]):
        selected_numeric_field = fid
        break
# Fallback to first number column
if not selected_numeric_field:
    for fid, f in main_field_ids.items():
        if f.data_type in ["Number", "Float", "Integer"]:
            selected_numeric_field = fid
            break
# Fall back to the first available column
if not selected_numeric_field and len(dataframes[main_record_set_id].columns) > 0:
    selected_numeric_field = dataframes[main_record_set_id].columns[0]

print(f"Using numeric field: {selected_numeric_field}")

threshold = dataframes[main_record_set_id][selected_numeric_field].astype(float).mean()

filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][selected_numeric_field].astype(float) > threshold]
print(f"Filtered records with {selected_numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{selected_numeric_field}_normalized"] = (filtered_df[selected_numeric_field].astype(float) - filtered_df[selected_numeric_field].astype(float).mean()) / filtered_df[selected_numeric_field].astype(float).std()
print(f"Normalized {selected_numeric_field} for filtered records:")
print(filtered_df[[selected_numeric_field, f"{selected_numeric_field}_normalized"]].head())

# Grouping by a key attribute, e.g., protein description or accession
group_field = None
for fid, f in main_field_ids.items():
    if any(x in (f.name or '').lower() for x in ["group", "class", "desc", "accession"]):
        group_field = fid
        break

if group_field and group_field in dataframes[main_record_set_id].columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field (before filtering)
plt.figure(figsize=(8, 4))
sns.histplot(dataframes[main_record_set_id][selected_numeric_field].astype(float), bins=30, kde=True)
plt.title(f"Distribution of {selected_numeric_field}")
plt.xlabel(selected_numeric_field)
plt.ylabel("Count")
plt.show()

# Scatter plot (if there are at least two numeric columns)
numeric_cols = [col for col in dataframes[main_record_set_id].columns if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col])]
if len(numeric_cols) >= 2:
    plt.figure(figsize=(6,6))
    sns.scatterplot(x=dataframes[main_record_set_id][numeric_cols[0]], y=dataframes[main_record_set_id][numeric_cols[1]])
    plt.xlabel(numeric_cols[0])
    plt.ylabel(numeric_cols[1])
    plt.title(f"Scatter plot of {numeric_cols[0]} vs {numeric_cols[1]}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains protein-level data from mass spectrometry experiments on extracellular vesicles from human mast cells, provided in tabular format.
- We reviewed available record sets, fields, and dynamically selected numeric fields for basic EDA and visualization, enabling further downstream statistical/proteomic analysis.
- The Croissant schema's strong typing and `@id` referencing enables robust programmatic access and reproducibility.